# Transformer from Scratch

In this notebook we implement the core Transformer architecture (Attention Is All You Need, Vaswani et al. 2017) and apply it to a **sentiment classification** task using the IMDB dataset.

## What we'll cover
1. Scaled Dot-Product Attention
2. Multi-Head Attention
3. Positional Encoding
4. Transformer Encoder Block
5. Text Classification Head
6. Training & Evaluation on IMDB

## 1. Imports & Setup

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchtext.datasets import IMDB
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Dataset Preparation

In [ ]:
# ----- Tokeniser & Vocabulary -----
tokenizer = get_tokenizer('basic_english')

def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizer(text)

# Build vocabulary from training split
train_iter = IMDB(split='train')
vocab = build_vocab_from_iterator(
    yield_tokens(train_iter),
    specials=['<unk>', '<pad>'],
    max_tokens=25_000
)
vocab.set_default_index(vocab['<unk>'])

VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab['<pad>']
print(f'Vocabulary size: {VOCAB_SIZE}')

In [ ]:
# ----- Collate & DataLoader -----
MAX_SEQ_LEN = 256
BATCH_SIZE  = 64

label_map = {'neg': 0, 'pos': 1}

def text_pipeline(text):
    tokens = tokenizer(text)[:MAX_SEQ_LEN]
    return vocab(tokens)

def collate_batch(batch):
    labels, token_ids = [], []
    for label, text in batch:
        labels.append(label_map[label])
        token_ids.append(torch.tensor(text_pipeline(text), dtype=torch.long))

    # Pad sequences to the same length within the batch
    token_ids = nn.utils.rnn.pad_sequence(
        token_ids, batch_first=True, padding_value=PAD_IDX
    )
    labels = torch.tensor(labels, dtype=torch.long)
    return token_ids.to(DEVICE), labels.to(DEVICE)

train_iter, test_iter = IMDB(split=('train', 'test'))
train_loader = DataLoader(list(train_iter), batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_batch)
test_loader  = DataLoader(list(test_iter),  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_batch)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

## 3. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query), **K** (Key), **V** (Value) are linear projections of the input.
- Dividing by $\sqrt{d_k}$ prevents vanishing gradients when $d_k$ is large.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q : (..., seq_q, d_k)
        K : (..., seq_k, d_k)
        V : (..., seq_k, d_v)
        mask: optional boolean mask; True positions are IGNORED.

    Returns:
        output : (..., seq_q, d_v)
        weights: (..., seq_q, seq_k)  -- useful for visualisation
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (..., seq_q, seq_k)

    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    weights = F.softmax(scores, dim=-1)   # (..., seq_q, seq_k)
    output  = torch.matmul(weights, V)    # (..., seq_q, d_v)
    return output, weights

## 4. Multi-Head Attention

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

where each $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$.

Running multiple attention heads in parallel lets the model jointly attend to information from different representation subspaces.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """(B, T, d_model) -> (B, heads, T, d_k)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)

        Q = self.split_heads(self.W_q(Q))  # (B, heads, T_q, d_k)
        K = self.split_heads(self.W_k(K))  # (B, heads, T_k, d_k)
        V = self.split_heads(self.W_v(V))  # (B, heads, T_k, d_k)

        # Expand mask for heads: (B, 1, 1, T_k)
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)

        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)  # (B, heads, T_q, d_k)

        # Concatenate heads
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(attn_out)

## 5. Positional Encoding

Since attention has no notion of order, we inject position information:

$$PE_{(pos, 2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)                    # (max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()   # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)                                   # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """x: (B, T, d_model)"""
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# Quick visualisation
pe_demo = PositionalEncoding(d_model=64, max_len=100)
pe_vals = pe_demo.pe.squeeze(0).numpy()  # (100, 64)

plt.figure(figsize=(12, 4))
plt.imshow(pe_vals.T, aspect='auto', cmap='RdBu')
plt.xlabel('Position'); plt.ylabel('Dimension')
plt.title('Positional Encoding heatmap (d_model=64)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 6. Feed-Forward Network & Encoder Block

In [ ]:
class FeedForward(nn.Module):
    """Position-wise FFN: two linear layers with ReLU in between."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    """
    One Transformer encoder layer:
      x -> MultiHeadAttn -> Add & Norm -> FFN -> Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads)
        self.ff      = FeedForward(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_key_padding_mask=None):
        # Self-attention sub-layer
        attn_out = self.attn(x, x, x, mask=src_key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward sub-layer
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

## 7. Transformer Classifier

In [ ]:
class TransformerClassifier(nn.Module):
    """
    Stacked Transformer encoder + mean-pooling classification head.
    """
    def __init__(
        self,
        vocab_size,
        num_classes,
        d_model=128,
        num_heads=4,
        num_layers=2,
        d_ff=256,
        max_len=512,
        dropout=0.1,
        pad_idx=0,
    ):
        super().__init__()
        self.pad_idx  = pad_idx
        self.embed    = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc  = PositionalEncoding(d_model, max_len, dropout)
        self.layers   = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, x):
        """
        x: (B, T)  -- token indices
        """
        # Padding mask: True where token == PAD  ->  those positions are masked out
        pad_mask = (x == self.pad_idx)             # (B, T)

        x = self.embed(x)                          # (B, T, d_model)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=pad_mask)

        # Mean-pool over non-padding positions
        mask_expanded = (~pad_mask).unsqueeze(-1).float()  # (B, T, 1)
        x = (x * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1)

        return self.classifier(x)

## 8. Training

In [ ]:
# ----- Hyperparameters -----
D_MODEL    = 128
NUM_HEADS  = 4
NUM_LAYERS = 2
D_FF       = 256
DROPOUT    = 0.1
LR         = 1e-3
EPOCHS     = 5

model = TransformerClassifier(
    vocab_size  = VOCAB_SIZE,
    num_classes = 2,
    d_model     = D_MODEL,
    num_heads   = NUM_HEADS,
    num_layers  = NUM_LAYERS,
    d_ff        = D_FF,
    max_len     = MAX_SEQ_LEN,
    dropout     = DROPOUT,
    pad_idx     = PAD_IDX,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for tokens, labels in loader:
        optimizer.zero_grad()
        logits = model(tokens)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for tokens, labels in loader:
        logits     = model(tokens)
        loss       = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    va_loss, va_acc = evaluate(model, test_loader, criterion)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train loss {tr_loss:.4f}  acc {tr_acc:.4f} | '
          f'Val loss {va_loss:.4f}  acc {va_acc:.4f} | '
          f'{elapsed:.1f}s')

## 9. Results & Learning Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('Transformer Classifier — IMDB Sentiment', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nFinal test accuracy: {history["val_acc"][-1]*100:.2f}%')

## 10. Attention Visualisation

Peek at what the first encoder layer attends to for a sample sentence.

In [ ]:
def get_attention_weights(model, tokens_tensor):
    """Return attention weights from the first encoder layer."""
    model.eval()
    with torch.no_grad():
        pad_mask = (tokens_tensor == model.pad_idx)
        x = model.embed(tokens_tensor)
        x = model.pos_enc(x)

        # Manually run the first layer and capture attention weights
        layer   = model.layers[0]
        Q = layer.attn.split_heads(layer.attn.W_q(x))
        K = layer.attn.split_heads(layer.attn.W_k(x))
        V = layer.attn.split_heads(layer.attn.W_v(x))

        mask = pad_mask.unsqueeze(1).unsqueeze(2) if pad_mask is not None else None
        d_k  = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        weights = F.softmax(scores, dim=-1)  # (B, heads, T, T)
    return weights.cpu()


sample_text  = "This movie was absolutely fantastic! The acting and story were top notch."
sample_tokens = torch.tensor(
    [text_pipeline(sample_text)[:MAX_SEQ_LEN]], dtype=torch.long
).to(DEVICE)

weights = get_attention_weights(model, sample_tokens)  # (1, heads, T, T)
words   = tokenizer(sample_text)[:MAX_SEQ_LEN]
T       = len(words)

fig, axes = plt.subplots(1, NUM_HEADS, figsize=(4 * NUM_HEADS, 4))
for h in range(NUM_HEADS):
    ax = axes[h]
    w  = weights[0, h, :T, :T].numpy()
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    ax.set_xticks(range(T)); ax.set_xticklabels(words, rotation=90, fontsize=7)
    ax.set_yticks(range(T)); ax.set_yticklabels(words, fontsize=7)
    ax.set_title(f'Head {h+1}')
    plt.colorbar(im, ax=ax)

plt.suptitle('Multi-Head Attention weights (Layer 1)', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Inference on Custom Text

In [ ]:
def predict_sentiment(text, model, vocab, tokenizer, device):
    model.eval()
    tokens = torch.tensor(
        [vocab(tokenizer(text)[:MAX_SEQ_LEN])], dtype=torch.long
    ).to(device)

    with torch.no_grad():
        logits = model(tokens)
        probs  = F.softmax(logits, dim=-1).squeeze()

    label = 'POSITIVE' if probs[1] > probs[0] else 'NEGATIVE'
    confidence = probs.max().item()
    print(f'  Text      : {text}')
    print(f'  Sentiment : {label}  (confidence: {confidence*100:.1f}%)')
    print()


samples = [
    "An absolute masterpiece. The director crafted a truly unforgettable experience.",
    "Terrible film. Boring plot, bad acting, and a complete waste of time.",
    "It was okay. Some good moments but nothing particularly memorable.",
    "One of the best movies I have seen in years. Highly recommended!",
]

print('--- Sentiment Predictions ---')
for s in samples:
    predict_sentiment(s, model, vocab, tokenizer, DEVICE)

## Summary

| Component | Details |
|---|---|
| Embedding | 25 000-token vocab, d_model=128 |
| Positional Encoding | Sinusoidal, max_len=256 |
| Encoder layers | 2 × EncoderBlock |
| Attention heads | 4 per layer |
| FFN hidden dim | 256 |
| Pooling | Masked mean-pool |
| Classifier | Linear → ReLU → Linear(2) |
| Optimiser | Adam + OneCycleLR |
| Dataset | IMDB (25k train / 25k test) |

### Key Takeaways
- **Scaled dot-product attention** allows every token to attend to every other token in O(T²) time.
- **Multi-head attention** enables the model to learn diverse attention patterns simultaneously.
- **Positional encoding** injects order information without recurrence.
- Even a small 2-layer Transformer can achieve strong sentiment classification accuracy.

### Extensions to Try
- Add more encoder layers or increase `d_model`.
- Replace mean-pooling with a `[CLS]` token (BERT-style).
- Use pre-trained word embeddings (GloVe / fastText).
- Swap the dataset for a multi-class classification problem.